In [163]:
# after merge do the feature removal and  selection

# Merging cc calls and billings 

In [164]:
# import pandas as pd
# import numpy as np
# import seaborn as sns
# import matplotlib.pyplot as plt
# import math
# import os
# from scipy.stats import pointbiserialr

# # ── YOUR EXISTING CODE ───────────────────────────────────────────────────────
# cc_calls    = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv')

# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# cc_calls['Call_Date']                = pd.to_datetime(cc_calls['Call_Date'],                dayfirst=True)
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(billings_df['Prospect_Renewal_Date'], dayfirst=True)

# merged_cc_calls = cc_calls.merge(billings_df, on='Co_Ref', how='inner')

# merged_cc_calls['Days_Before_Renewal'] = (
#     merged_cc_calls['Prospect_Renewal_Date'] - merged_cc_calls['Call_Date']
# ).dt.days

# merged_cc_calls = merged_cc_calls[
#     (merged_cc_calls['Days_Before_Renewal'] >= 14) &
#     (merged_cc_calls['Days_Before_Renewal'] <= 45)
# ]

# rows_to_drop = merged_cc_calls[merged_cc_calls['Prospect_Outcome'] == 'Open'].index
# merged_cc_calls.drop(rows_to_drop, inplace=True)

# merged_cc_calls = merged_cc_calls.sort_values(
#     by=['Co_Ref', 'Call_Date'], ascending=[True, True]
# )
# # ─────────────────────────────────────────────────────────────────────────────

# print('Shape after merge + filter:', merged_cc_calls.shape)
# print('Prospect_Outcome counts:')
# print(merged_cc_calls['Prospect_Outcome'].value_counts())

## Merging  bill + cc_calls (inner)

In [165]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# cc_calls = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv')

# # Keep only required billing columns (1 row per Co_Ref)
# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# # Convert dates
# cc_calls['Call_Date'] = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # ── STEP 1: MERGE (BILLING AS BASE) ─────────────────────────────────
# merged = billings_df.merge(cc_calls, on='Co_Ref', how='inner')

# print("Shape after merge:", merged.shape)

# # ── STEP 2: CALCULATE DAYS BEFORE RENEWAL ───────────────────────────
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# # ── STEP 3: APPLY 14–45 DAY FILTER ──────────────────────────────────
# filtered = merged[
#     (merged['Days_Before_Renewal'] >= 14) &
#     (merged['Days_Before_Renewal'] <= 45)
# ]

# print("Shape after 14–45 day filter:", filtered.shape)

# # ── STEP 4: SORT FOR CLEAR VIEW ─────────────────────────────────────
# filtered = filtered.sort_values(by=['Co_Ref', 'Call_Date'])

# # ── STEP 5: SHOW MULTIPLE CALLS PER CUSTOMER ────────────────────────
# print("\nExample (1 billing → multiple filtered calls):\n")

# for i, (coref, group) in enumerate(filtered.groupby('Co_Ref')):
#     print(f"Co_Ref: {coref}")
    
#     print("Billing Info:")
#     print(group[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']].iloc[0])
    
#     print("\nFiltered Calls (14–45 days before renewal):")
#     print(group[['Call_Date', 'Days_Before_Renewal']])
    
#     print("-" * 50)
    
#     if i == 2:
#         break

# # ── STEP 6: COUNT FILTERED CALLS ────────────────────────────────────
# call_counts = filtered.groupby('Co_Ref')['Call_Date'].count()

# print("\nTop customers by number of filtered calls:")
# print(call_counts.sort_values(ascending=False).head(10))

# # ── STEP 7: CUSTOMERS WITH NO CALLS IN WINDOW ───────────────────────
# customers_with_calls = filtered['Co_Ref'].nunique()
# total_customers = billings_df['Co_Ref'].nunique()

# print(f"\nCustomers with calls in 14–45 day window: {customers_with_calls}")
# print(f"Customers with NO calls in this window: {total_customers - customers_with_calls}")

# call_distribution = call_counts.value_counts().sort_index()

# print("\nDistribution of number of calls per customer:\n")

# for calls, num_customers in call_distribution.items():
#     print(f"{calls} calls → {num_customers} customers")

# print(filtered)


## Merging  bill + renew (inner)

### wihtout left 

In [166]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
# billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# renewal_calls['Call_Date'] = pd.to_datetime(
#     renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
# )

# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Remove invalid calls
# renewal_calls = renewal_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# # ────────────────────────────────────────────────────────────────────
# billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

# billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
#     'Prospect_Renewal_Date'
# ].shift(1)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: MERGE
# # ────────────────────────────────────────────────────────────────────
# merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: FILTER CURRENT CYCLE
# # ────────────────────────────────────────────────────────────────────
# merged = merged[
#     (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
#     (
#         (merged['Prev_Renewal_Date'].isna()) |
#         (merged['Call_Date'] > merged['Prev_Renewal_Date'])
#     )
# ]


# # merged = merged[
# #     (merged['Call_Date'].isna()) |   # 👈 FIX
# #     (
# #         (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
# #         (
# #             (merged['Prev_Renewal_Date'].isna()) |
# #             (merged['Call_Date'] > merged['Prev_Renewal_Date'])
# #         )
# #     )
# # ]

# print("Shape after cycle filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: APPLY 14–45 DAY WINDOW
# # ────────────────────────────────────────────────────────────────────
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# merged = merged[
#     (merged['Days_Before_Renewal'] >= 14) &
#     (merged['Days_Before_Renewal'] <= 45)
# ]

# print("Shape after 14–45 day filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 5: REMOVE DUPLICATES (IMPORTANT)
# # ────────────────────────────────────────────────────────────────────
# merged = merged.drop_duplicates(
#     subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 6: DEBUG – HIGH ACTIVITY CUSTOMERS (>10 ROWS)
# # ────────────────────────────────────────────────────────────────────
# counts = merged['Co_Ref'].value_counts()
# valid_coref = counts[counts > 5].index

# filtered = merged[merged['Co_Ref'].isin(valid_coref)]

# filtered = filtered.sort_values(
#     ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]

# # Save debug file
# output.to_csv('./sample_filtered_coref.csv', index=False)
# print("Saved debug file: ./sample_filtered_coref.csv")

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 7: SAVE FINAL DATASET
# # ────────────────────────────────────────────────────────────────────
# merged.to_csv("../../dataset/05_merged/br_merged.csv", index=False)

# print("Saved final dataset: br_merged.csv")

### with left

In [152]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)

billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# ────────────────────────────────────────────────────────────────────
billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: MERGE (RENEWAL CALLS)
# ────────────────────────────────────────────────────────────────────
merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: FILTER CURRENT CYCLE (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged = merged[
    (merged['Call_Date'].isna()) |   # 👈 IMPORTANT
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]

print("Shape after cycle filter:", merged.shape)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: APPLY 14–45 WINDOW (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |   # 👈 IMPORTANT
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]

print("Shape after 14–45 day filter:", merged.shape)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 5: REMOVE DUPLICATES
# ────────────────────────────────────────────────────────────────────
merged = merged.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 6: DEBUG
# ────────────────────────────────────────────────────────────────────
counts = merged['Co_Ref'].value_counts()
valid_coref = counts[counts > 5].index

filtered = merged[merged['Co_Ref'].isin(valid_coref)]

filtered = filtered.sort_values(
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]

output.to_csv('./sample_filtered_renewal_coref.csv', index=False)
print("Saved debug file: ./sample_filtered_renewal_coref.csv")

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 7: SAVE FINAL DATASET
# ────────────────────────────────────────────────────────────────────
merged.to_csv("../../dataset/05_merged/br_merged.csv", index=False)

print("Saved final dataset: br_merged.csv")
print(merged)

C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\1833760617.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on im

Shape after cycle filter: (141276, 90)
Shape after 14–45 day filter: (62443, 91)
Saved debug file: ./sample_filtered_renewal_coref.csv
Saved final dataset: br_merged.csv
        Co_Ref Renewal_Month Discount_Amount  Sustainability_Score  \
0       AA0106    01-02-2023               0                   8.0   
1       AA0106    01-02-2024               0                   8.0   
4       AA0584    01-05-2024               0                   9.5   
13      AA0641    01-11-2024               0                   8.0   
19      AA0750    01-12-2024               0                   9.5   
...        ...           ...             ...                   ...   
441060  ZZ9867    01-09-2023               0                   9.5   
441061  ZZ9867    01-09-2024               0                   9.5   
441062  ZZ9867    01-09-2025               0                   9.5   
441064  ZZ9952    01-08-2024               0                   9.0   
441068  ok0108    01-07-2024               0                

# CC Calls + bill

### wihtout left 

In [153]:


# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# cc_calls    = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# cc_calls['Call_Date'] = pd.to_datetime(
#     cc_calls['Call_Date'], dayfirst=True, errors='coerce'
# )

# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Remove invalid calls
# cc_calls = cc_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# # ────────────────────────────────────────────────────────────────────
# billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

# billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
#     'Prospect_Renewal_Date'
# ].shift(1)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: MERGE
# # ────────────────────────────────────────────────────────────────────
# merged = billings_df.merge(cc_calls, on='Co_Ref', how='left')

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: FILTER CURRENT CYCLE (LEFT SAFE)
# # ────────────────────────────────────────────────────────────────────
# # merged = merged[
# #     (merged['Call_Date'].isna()) |   # 👈 IMPORTANT FIX
# #     (
# #         (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
# #         (
# #             (merged['Prev_Renewal_Date'].isna()) |
# #             (merged['Call_Date'] > merged['Prev_Renewal_Date'])
# #         )
# #     )
# # ]

# merged = merged[
#     (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
#     (
#         (merged['Prev_Renewal_Date'].isna()) |
#         (merged['Call_Date'] > merged['Prev_Renewal_Date'])
#     )
# ]

# print("Shape after cycle filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: APPLY 14–45 DAY WINDOW (LEFT SAFE)
# # ────────────────────────────────────────────────────────────────────
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# merged = merged[
#     (merged['Call_Date'].isna()) |   # 👈 IMPORTANT FIX
#     (
#         (merged['Days_Before_Renewal'] >= 14) &
#         (merged['Days_Before_Renewal'] <= 45)
#     )
# ]

# print("Shape after 14–45 day filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 5: REMOVE DUPLICATES
# # ────────────────────────────────────────────────────────────────────
# merged = merged.drop_duplicates(
#     subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 6: DEBUG – HIGH ACTIVITY CUSTOMERS (>5 ROWS)
# # ────────────────────────────────────────────────────────────────────
# counts = merged['Co_Ref'].value_counts()
# valid_coref = counts[counts > 5].index

# filtered = merged[merged['Co_Ref'].isin(valid_coref)]

# filtered = filtered.sort_values(
#     ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]

# # Save debug file
# output.to_csv('./sample_filtered_cc_coref.csv', index=False)
# print("Saved debug file: ./sample_filtered_cc_coref.csv")

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 7: SAVE FINAL DATASET
# # ────────────────────────────────────────────────────────────────────
# merged.to_csv("../../dataset/05_merged/bcc_merged.csv", index=False)

# print("Saved final dataset: bcc_merged.csv")




### with left 

In [154]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
cc_calls    = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# ── DATE CONVERSION ─────────────────────────────────────────────────
cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)

billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
cc_calls = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# ────────────────────────────────────────────────────────────────────
billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: MERGE
# ────────────────────────────────────────────────────────────────────
merged = billings_df.merge(cc_calls, on='Co_Ref', how='left')

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: FILTER CURRENT CYCLE (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged = merged[
    (merged['Call_Date'].isna()) |   # 👈 IMPORTANT FIX
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]

print("Shape after cycle filter:", merged.shape)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: APPLY 14–45 DAY WINDOW (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |   # 👈 IMPORTANT FIX
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]

print("Shape after 14–45 day filter:", merged.shape)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 5: REMOVE DUPLICATES
# ────────────────────────────────────────────────────────────────────
merged = merged.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 6: DEBUG – HIGH ACTIVITY CUSTOMERS (>5 ROWS)
# ────────────────────────────────────────────────────────────────────
counts = merged['Co_Ref'].value_counts()
valid_coref = counts[counts > 5].index

filtered = merged[merged['Co_Ref'].isin(valid_coref)]

filtered = filtered.sort_values(
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]

# Save debug file
output.to_csv('./sample_filtered_cc_coref.csv', index=False)
print("Saved debug file: ./sample_filtered_cc_coref.csv")

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 7: SAVE FINAL DATASET
# ────────────────────────────────────────────────────────────────────
merged.to_csv("../../dataset/05_merged/bcc_merged.csv", index=False)

print("Saved final dataset: bcc_merged.csv")

Shape after cycle filter: (104564, 87)
Shape after 14–45 day filter: (82695, 88)
Saved debug file: ./sample_filtered_cc_coref.csv
Saved final dataset: bcc_merged.csv


In [155]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# cc_calls    = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# cc_calls['Call_Date'] = pd.to_datetime(
#     cc_calls['Call_Date'], dayfirst=True, errors='coerce'
# )

# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Remove invalid calls
# cc_calls = cc_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# # ────────────────────────────────────────────────────────────────────
# billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

# billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
#     'Prospect_Renewal_Date'
# ].shift(1)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: MERGE (CC CALLS)
# # ────────────────────────────────────────────────────────────────────
# merged = billings_df.merge(cc_calls, on='Co_Ref', how='left')

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: FILTER CURRENT CYCLE
# # ────────────────────────────────────────────────────────────────────
# merged = merged[
#     (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
#     (
#         (merged['Prev_Renewal_Date'].isna()) |
#         (merged['Call_Date'] > merged['Prev_Renewal_Date'])
#     )
# ]

# print("Shape after cycle filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: APPLY 14–45 DAY WINDOW
# # ────────────────────────────────────────────────────────────────────
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# merged = merged[
#     (merged['Days_Before_Renewal'] >= 14) &
#     (merged['Days_Before_Renewal'] <= 45)
# ]

# print("Shape after 14–45 day filter:", merged.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 5: REMOVE DUPLICATES
# # ────────────────────────────────────────────────────────────────────
# merged = merged.drop_duplicates(
#     subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 6: DEBUG – HIGH ACTIVITY CUSTOMERS (>5 ROWS)
# # ────────────────────────────────────────────────────────────────────
# counts = merged['Co_Ref'].value_counts()
# valid_coref = counts[counts > 5].index

# filtered = merged[merged['Co_Ref'].isin(valid_coref)]

# filtered = filtered.sort_values(
#     ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
# )

# output = filtered[['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']]

# # Save debug file
# output.to_csv('./sample_filtered_cc_coref.csv', index=False)
# print("Saved debug file: ./sample_filtered_cc_coref.csv")

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 7: SAVE FINAL DATASET
# # ────────────────────────────────────────────────────────────────────
# merged.to_csv("../../dataset/05_merged/bcc_merged.csv", index=False)

# print("Saved final dataset: bcc_merged.csv")

In [156]:
# import pandas as pd

# # ── SAMPLE DATA ─────────────────────────────────────────────────────
# billings_df = pd.DataFrame({
#     'Co_Ref': ['A', 'A'],
#     'Prospect_Renewal_Date': ['12-01-2025', '12-01-2026']
# })

# renewal_calls = pd.DataFrame({
#     'Co_Ref': ['A','A','A','A','A','A'],
#     'Call_Date': [
#         '19-12-2024',
#         '20-12-2024',
#         '21-12-2024',
#         '19-12-2025',
#         '19-12-2025',
#         '19-12-2025'
#     ]
# })

# # ── CONVERT DATES ───────────────────────────────────────────────────
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True
# )

# renewal_calls['Call_Date'] = pd.to_datetime(
#     renewal_calls['Call_Date'], dayfirst=True
# )

# # ── MERGE ───────────────────────────────────────────────────────────
# merged = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# # Keep only calls BEFORE renewal
# merged = merged[
#     (merged['Call_Date'].isna()) |
#     (merged['Call_Date'] <= merged['Prospect_Renewal_Date'])
# ]

# # Days before renewal
# merged['Days_Before_Renewal'] = (
#     merged['Prospect_Renewal_Date'] - merged['Call_Date']
# ).dt.days

# # Flag 14–45 calls
# merged['renewal_14_45_flag'] = (
#     (merged['Days_Before_Renewal'] >= 14) &
#     (merged['Days_Before_Renewal'] <= 45)
# ).astype(int)

# # ── AGGREGATION (ONLY BY Co_Ref - same as your code) ────────────────
# agg_calls = merged.groupby('Co_Ref').agg(
#     renewal_call_count_14_45=('renewal_14_45_flag', 'sum')
# ).reset_index()

# # ── MERGE BACK ──────────────────────────────────────────────────────
# final_df = billings_df.merge(agg_calls, on='Co_Ref', how='left')

# # Fill missing
# final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)

# # ── OUTPUT ─────────────────────────────────────────────────────────
# print(final_df)

# Merged  (billing + renew (left ) + cc (left))


In [157]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)

cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)

billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# ────────────────────────────────────────────────────────────────────
billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: MERGE RENEWAL CALLS (ALL COLS)
# ────────────────────────────────────────────────────────────────────
merged = billings_df.merge(
    renewal_calls,
    on='Co_Ref',
    how='left',
    suffixes=('', '_renewal')
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: FILTER RENEWAL CALLS (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Call_Date'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date'] > merged['Prev_Renewal_Date'])
        )
    )
]

# 14–45 filter
merged['Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date']
).dt.days

merged = merged[
    (merged['Call_Date'].isna()) |
    (
        (merged['Days_Before_Renewal'] >= 14) &
        (merged['Days_Before_Renewal'] <= 45)
    )
]

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: MERGE CC CALLS (ALL COLS)
# ────────────────────────────────────────────────────────────────────
merged = merged.merge(
    cc_calls,
    on='Co_Ref',
    how='left',
    suffixes=('', '_cc')
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 5: FILTER CC CALLS (LEFT SAFE)
# ────────────────────────────────────────────────────────────────────
merged = merged[
    (merged['Call_Date_cc'].isna()) |
    (
        (merged['Call_Date_cc'] <= merged['Prospect_Renewal_Date']) &
        (
            (merged['Prev_Renewal_Date'].isna()) |
            (merged['Call_Date_cc'] > merged['Prev_Renewal_Date'])
        )
    )
]

# 14–45 filter for cc
merged['cc_Days_Before_Renewal'] = (
    merged['Prospect_Renewal_Date'] - merged['Call_Date_cc']
).dt.days

merged = merged[
    (merged['Call_Date_cc'].isna()) |
    (
        (merged['cc_Days_Before_Renewal'] >= 14) &
        (merged['cc_Days_Before_Renewal'] <= 45)
    )
]

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 6: REMOVE DUPLICATES
# ────────────────────────────────────────────────────────────────────
# merged = merged.drop_duplicates()

# ────────────────────────────────────────────────────────────────────
# ✅ FINAL OUTPUT
# ────────────────────────────────────────────────────────────────────
print("Final dataset shape:", merged.shape)
print(merged.head())

# Save
merged.to_csv("../../dataset/05_merged/full_raw_billing_calls.csv", index=False)

print("Saved: full_raw_billing_calls.csv")

C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\906988083.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on imp

Final dataset shape: (44481, 124)
   Co_Ref Renewal_Month Discount_Amount  Sustainability_Score  \
0  AA0106    01-02-2023               0                   8.0   
1  AA0106    01-02-2024               0                   8.0   
4  AA0641    01-11-2024               0                   8.0   
5  AA0641    01-11-2024               0                   8.0   
6  AA0641    01-11-2024               0                   8.0   

   Total_Renewal_Score_New  Last_Years_Price  Auto_Renewal_Score  \
0                     34.0            1119.0                   8   
1                     34.0               NaN                   8   
4                     44.0             799.0                   9   
5                     44.0             799.0                   9   
6                     44.0             799.0                   9   

   Status_Scores  Anchoring_Score  Tenure_Scores  ...  \
0              0              8.5            9.5  ...   
1              0              8.5            9.5  ..

In [158]:
import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)

cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)

billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# ────────────────────────────────────────────────────────────────────
billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: PROCESS RENEWAL CALLS
# ────────────────────────────────────────────────────────────────────
br = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# Cycle filter (LEFT SAFE)
br = br[
    (br['Call_Date'].isna()) |
    (
        (br['Call_Date'] <= br['Prospect_Renewal_Date']) &
        (
            (br['Prev_Renewal_Date'].isna()) |
            (br['Call_Date'] > br['Prev_Renewal_Date'])
        )
    )
]

# 14–45 filter
br['Days_Before_Renewal'] = (
    br['Prospect_Renewal_Date'] - br['Call_Date']
).dt.days

br = br[
    (br['Call_Date'].isna()) |
    (
        (br['Days_Before_Renewal'] >= 14) &
        (br['Days_Before_Renewal'] <= 45)
    )
]

# Remove duplicates
br = br.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

# Aggregate renewal calls
br_agg = br.groupby(['Co_Ref', 'Prospect_Renewal_Date']).agg(
    renewal_call_count_14_45=('Call_Date', 'count')
).reset_index()

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: PROCESS CC CALLS
# ────────────────────────────────────────────────────────────────────
bcc = billings_df.merge(cc_calls, on='Co_Ref', how='left')

# Cycle filter (LEFT SAFE)
bcc = bcc[
    (bcc['Call_Date'].isna()) |
    (
        (bcc['Call_Date'] <= bcc['Prospect_Renewal_Date']) &
        (
            (bcc['Prev_Renewal_Date'].isna()) |
            (bcc['Call_Date'] > bcc['Prev_Renewal_Date'])
        )
    )
]

# 14–45 filter
bcc['Days_Before_Renewal'] = (
    bcc['Prospect_Renewal_Date'] - bcc['Call_Date']
).dt.days

bcc = bcc[
    (bcc['Call_Date'].isna()) |
    (
        (bcc['Days_Before_Renewal'] >= 14) &
        (bcc['Days_Before_Renewal'] <= 45)
    )
]

# Remove duplicates
bcc = bcc.drop_duplicates(
    subset=['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
)

# Aggregate cc calls
bcc_agg = bcc.groupby(['Co_Ref', 'Prospect_Renewal_Date']).agg(
    cc_call_count_14_45=('Call_Date', 'count')
).reset_index()

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: FINAL MERGE (ALL FEATURES)
# ────────────────────────────────────────────────────────────────────
final_df = billings_df.merge(
    br_agg, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)

print("Before" )

final_df = final_df.merge(
    bcc_agg, on=['Co_Ref', 'Prospect_Renewal_Date'], how='left'
)

# Fill missing (no calls → 0)
final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)
final_df['cc_call_count_14_45']      = final_df['cc_call_count_14_45'].fillna(0)

# ────────────────────────────────────────────────────────────────────
# ✅ FINAL OUTPUT
# ────────────────────────────────────────────────────────────────────
print("Final dataset shape:", final_df.shape)
print(final_df.head())

# Save
final_df.to_csv("../../dataset/05_merged/final_billing_calls.csv", index=False)

print("Saved final dataset: final_billing_calls.csv")



# sample output 
# ────────────────────────────────────────────────────────────────────
# ✅ SAMPLE OUTPUT (DEBUG VIEW)
# ────────────────────────────────────────────────────────────────────

# ────────────────────────────────────────────────────────────────────
# ✅ SAMPLE WITH ACTUAL CALL DATES (RENEWAL + CC)
# ────────────────────────────────────────────────────────────────────

# Step 1: pick active customers (same logic)
counts = final_df.groupby('Co_Ref')[
    ['renewal_call_count_14_45', 'cc_call_count_14_45']
].sum()

counts['total'] = counts.sum(axis=1)
valid_coref = counts[counts['total'] >= 0].index

# Step 2: filter original (NON-AGGREGATED) datasets
br_sample = br[br['Co_Ref'].isin(valid_coref)][
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
].rename(columns={'Call_Date': 'Renewal_Call_Date'})

bcc_sample = bcc[bcc['Co_Ref'].isin(valid_coref)][
    ['Co_Ref', 'Prospect_Renewal_Date', 'Call_Date']
].rename(columns={'Call_Date': 'CC_Call_Date'})

# Step 3: merge both (outer to see all)
sample_calls = pd.merge(
    br_sample,
    bcc_sample,
    on=['Co_Ref', 'Prospect_Renewal_Date'],
    how='outer'
)

# Step 4: sort
sample_calls = sample_calls.sort_values(
    ['Co_Ref', 'Prospect_Renewal_Date', 'Renewal_Call_Date', 'CC_Call_Date']
)

# Step 5: print
print("\nSample with Call Dates:")
print(sample_calls.head(30))

# Step 6: save
sample_calls.to_csv('./sample_call_dates.csv', index=False)

print("Saved: ./sample_call_dates.csv")


C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\3801687104.py:4: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on im

Before
Final dataset shape: (122082, 57)
   Co_Ref Renewal_Month Discount_Amount  Sustainability_Score  \
0  AA0106    01-02-2023               0                   8.0   
1  AA0106    01-02-2024               0                   8.0   
2  AA0584    01-05-2023               0                   9.5   
3  AA0584    01-05-2024               0                   9.5   
4  AA0584    01-05-2025               0                   9.5   

   Total_Renewal_Score_New  Last_Years_Price  Auto_Renewal_Score  \
0                     34.0            1119.0                   8   
1                     34.0               NaN                   8   
2                     45.5             489.0                   9   
3                     45.5             579.0                   9   
4                     45.5             599.0                   9   

   Status_Scores  Anchoring_Score  Tenure_Scores  ... Last_Renewal Last_Band  \
0              0              8.5            9.5  ...          NaN       NaN   

# FINAL MERGED CODE

In [159]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
# cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# print("Billing Shape:", billings_df.shape)
# print("cc_calls Shape:", cc_calls.shape)
# print("Renwal Shape:", renewal_calls.shape)

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# renewal_calls['Call_Date'] = pd.to_datetime(
#     renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
# )

# cc_calls['Call_Date'] = pd.to_datetime(
#     cc_calls['Call_Date'], dayfirst=True, errors='coerce'
# )

# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Remove invalid calls
# renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
# cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# # ────────────────────────────────────────────────────────────────────
# billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

# billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
#     'Prospect_Renewal_Date'
# ].shift(1)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: RENEWAL CALLS (KEEP FULL ROW)
# # ────────────────────────────────────────────────────────────────────
# br = billings_df.merge(
#     renewal_calls,
#     on='Co_Ref',
#     how='left',
#     suffixes=('', '_renewal')
# )

# # Cycle filter
# br = br[
#     (br['Call_Date'] <= br['Prospect_Renewal_Date']) &
#     (
#         (br['Prev_Renewal_Date'].isna()) |
#         (br['Call_Date'] > br['Prev_Renewal_Date'])
#     )
# ]

# # 14–45 filter
# br['Days_Before_Renewal'] = (
#     br['Prospect_Renewal_Date'] - br['Call_Date']
# ).dt.days

# br = br[
#     (br['Days_Before_Renewal'] >= 14) &
#     (br['Days_Before_Renewal'] <= 45)
# ]

# # 👉 TAKE FULL ROW OF LATEST CALL
# br_latest = br.sort_values('Call_Date').groupby(
#     ['Co_Ref', 'Prospect_Renewal_Date']
# ).tail(1)

# # Rename columns to avoid clash
# br_latest = br_latest.add_suffix('_renewal')

# # Restore keys
# br_latest = br_latest.rename(columns={
#     'Co_Ref_renewal': 'Co_Ref',
#     'Prospect_Renewal_Date_renewal': 'Prospect_Renewal_Date'
# })

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: CC CALLS (KEEP FULL ROW)
# # ────────────────────────────────────────────────────────────────────
# bcc = billings_df.merge(
#     cc_calls,
#     on='Co_Ref',
#     how='left',
#     suffixes=('', '_cc')
# )

# # Cycle filter
# bcc = bcc[
#     (bcc['Call_Date'] <= bcc['Prospect_Renewal_Date']) &
#     (
#         (bcc['Prev_Renewal_Date'].isna()) |
#         (bcc['Call_Date'] > bcc['Prev_Renewal_Date'])
#     )
# ]

# # 14–45 filter
# bcc['Days_Before_Renewal'] = (
#     bcc['Prospect_Renewal_Date'] - bcc['Call_Date']
# ).dt.days

# bcc = bcc[
#     (bcc['Days_Before_Renewal'] >= 14) &
#     (bcc['Days_Before_Renewal'] <= 45)
# ]

# # 👉 TAKE FULL ROW OF LATEST CALL
# bcc_latest = bcc.sort_values('Call_Date').groupby(
#     ['Co_Ref', 'Prospect_Renewal_Date']
# ).tail(1)

# # Rename columns
# bcc_latest = bcc_latest.add_suffix('_cc')

# bcc_latest = bcc_latest.rename(columns={
#     'Co_Ref_cc': 'Co_Ref',
#     'Prospect_Renewal_Date_cc': 'Prospect_Renewal_Date'
# })

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: FINAL MERGE (ALL COLS)
# # ────────────────────────────────────────────────────────────────────
# final_df = billings_df.merge(
#     br_latest,
#     on=['Co_Ref', 'Prospect_Renewal_Date'],
#     how='left'
# )

# final_df = final_df.merge(
#     bcc_latest,
#     on=['Co_Ref', 'Prospect_Renewal_Date'],
#     how='left'
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ OUTPUT
# # ────────────────────────────────────────────────────────────────────
# print("Final dataset shape:", final_df.shape)
# print(final_df.head())

# # Save
# final_df.to_csv("../../dataset/05_merged/final_latest_calls_fullcols.csv", index=False)

# print("Saved: final_latest_calls_fullcols.csv")





import pandas as pd

# ── LOAD DATA ───────────────────────────────────────────────────────
renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

print("Billing Shape:", billings_df.shape)
print("CC Calls Shape:", cc_calls.shape)
print("Renewal Calls Shape:", renewal_calls.shape)

# ── DATE CONVERSION ─────────────────────────────────────────────────
renewal_calls['Call_Date'] = pd.to_datetime(
    renewal_calls['Call_Date'], dayfirst=True, errors='coerce'
)

cc_calls['Call_Date'] = pd.to_datetime(
    cc_calls['Call_Date'], dayfirst=True, errors='coerce'
)

billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
    billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
)

# Remove invalid calls
renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 1: SORT & CREATE PREVIOUS RENEWAL
# ────────────────────────────────────────────────────────────────────
billings_df = billings_df.sort_values(['Co_Ref', 'Prospect_Renewal_Date'])

billings_df['Prev_Renewal_Date'] = billings_df.groupby('Co_Ref')[
    'Prospect_Renewal_Date'
].shift(1)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 2: RENEWAL CALLS (LATEST ONLY, NO DUPLICATE BILLING COLS)
# ────────────────────────────────────────────────────────────────────
br = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prev_Renewal_Date']].merge(
    renewal_calls,
    on='Co_Ref',
    how='left'
)

# Cycle filter
br = br[
    (br['Call_Date'] <= br['Prospect_Renewal_Date']) &
    (
        (br['Prev_Renewal_Date'].isna()) |
        (br['Call_Date'] > br['Prev_Renewal_Date'])
    )
]

# 14–45 filter
br['Days_Before_Renewal'] = (
    br['Prospect_Renewal_Date'] - br['Call_Date']
).dt.days

br = br[
    (br['Days_Before_Renewal'] >= 14) &
    (br['Days_Before_Renewal'] <= 45)
]

# 👉 TAKE LATEST CALL ROW
br_latest = br.sort_values('Call_Date').groupby(
    ['Co_Ref', 'Prospect_Renewal_Date']
).tail(1)

# Keep ONLY renewal call columns
renewal_cols = [col for col in renewal_calls.columns if col != 'Co_Ref']

br_latest = br_latest[
    ['Co_Ref', 'Prospect_Renewal_Date'] + renewal_cols
]

# Rename (avoid conflicts)
br_latest = br_latest.rename(
    columns={col: col + '_renewal' for col in renewal_cols if col != 'Co_Ref'}
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 3: CC CALLS (LATEST ONLY, NO DUPLICATE BILLING COLS)
# ────────────────────────────────────────────────────────────────────
bcc = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prev_Renewal_Date']].merge(
    cc_calls,
    on='Co_Ref',
    how='left'
)

# Cycle filter
bcc = bcc[
    (bcc['Call_Date'] <= bcc['Prospect_Renewal_Date']) &
    (
        (bcc['Prev_Renewal_Date'].isna()) |
        (bcc['Call_Date'] > bcc['Prev_Renewal_Date'])
    )
]

# 14–45 filter
bcc['Days_Before_Renewal'] = (
    bcc['Prospect_Renewal_Date'] - bcc['Call_Date']
).dt.days

bcc = bcc[
    (bcc['Days_Before_Renewal'] >= 14) &
    (bcc['Days_Before_Renewal'] <= 45)
]

# 👉 TAKE LATEST CALL ROW
bcc_latest = bcc.sort_values('Call_Date').groupby(
    ['Co_Ref', 'Prospect_Renewal_Date']
).tail(1)

# Keep ONLY cc call columns
cc_cols = [col for col in cc_calls.columns if col != 'Co_Ref']

bcc_latest = bcc_latest[
    ['Co_Ref', 'Prospect_Renewal_Date'] + cc_cols
]

# Rename
bcc_latest = bcc_latest.rename(
    columns={col: col + '_cc' for col in cc_cols if col != 'Co_Ref'}
)

# ────────────────────────────────────────────────────────────────────
# ✅ STEP 4: FINAL MERGE (CLEAN)
# ────────────────────────────────────────────────────────────────────
final_df = billings_df.merge(
    br_latest,
    on=['Co_Ref', 'Prospect_Renewal_Date'],
    how='left'
)

final_df = final_df.merge(
    bcc_latest,
    on=['Co_Ref', 'Prospect_Renewal_Date'],
    how='left'
)

# ────────────────────────────────────────────────────────────────────
# ✅ OUTPUT
# ────────────────────────────────────────────────────────────────────
print("Final dataset shape:", final_df.shape)
print(final_df.head())

# Save
final_df.to_csv("../../dataset/05_merged/final_latest_calls_fullcols.csv", index=False)

print("Saved: final_latest_calls_fullcols.csv")

C:\Users\NidharsanVelmurugan\AppData\Local\Temp\ipykernel_15232\2101287718.py:156: DtypeWarning: Columns (0: Churn_Category, 1: Complaint_Category, 2: Customer_Reaction_Category, 3: Agent_Renewal_Pitch_Category, 4: Customer_Renewal_Response_Category, 5: Agent_Response_Category, 6: Membership_Renewal_Decision, 7: Serious_Complaint, 8: Other_Complaint, 9: Discussion_on_Price_Increase, 10: Renewal_Impact_Due_to_Price_Increase, 11: Discount_or_Waiver_Requested, 12: Call_Reschedule_Request, 13: Agent_Flagged_Membership_Status_Alert, 14: Agent_Renewal_Initiation, 15: Explicit_Competitor_Mention, 16: Explicit_Switching_Intent, 17: Mentioned_Competitors, 18: Price_Switching_Mentioned, 19: Competitor_Benefits_Mentioned, 20: Topic_Introduced_By, 21: Percentage_Price_Increase_Mentioned, 22: Monetary_Price_Increase_Mentioned, 23: Price_Range_Mentioned, 24: Customer_Asked_For_Justification, 25: Customer_Response, 26: Desire_To_Cancel, 27: Discount_Offered) have mixed types. Specify dtype option on 

Billing Shape: (122082, 54)
CC Calls Shape: (32789, 33)
Renewal Calls Shape: (157722, 36)
Final dataset shape: (122082, 122)
   Co_Ref Renewal_Month Discount_Amount  Sustainability_Score  \
0  AA0106    01-02-2023               0                   8.0   
1  AA0106    01-02-2024               0                   8.0   
2  AA0584    01-05-2023               0                   9.5   
3  AA0584    01-05-2024               0                   9.5   
4  AA0584    01-05-2025               0                   9.5   

   Total_Renewal_Score_New  Last_Years_Price  Auto_Renewal_Score  \
0                     34.0            1119.0                   8   
1                     34.0               NaN                   8   
2                     45.5             489.0                   9   
3                     45.5             579.0                   9   
4                     45.5             599.0                   9   

   Status_Scores  Anchoring_Score  Tenure_Scores  ...  \
0              0  

In [160]:
# import pandas as pd

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: LOAD FILES
# # ────────────────────────────────────────────────────────────────────
# br = pd.read_csv("../../dataset/05_merged/br_merged.csv")     # renewal
# bcc = pd.read_csv("../../dataset/05_merged/bcc_merged.csv")   # cc
# billings_df = pd.read_csv("../../dataset/02_basic_clean/billings.csv", low_memory=False)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: DATE CONVERSION
# # ────────────────────────────────────────────────────────────────────
# br['Prospect_Renewal_Date'] = pd.to_datetime(br['Prospect_Renewal_Date'], dayfirst=True, errors='coerce')
# bcc['Prospect_Renewal_Date'] = pd.to_datetime(bcc['Prospect_Renewal_Date'], dayfirst=True, errors='coerce')

# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: CREATE RENEWAL FEATURES
# # ────────────────────────────────────────────────────────────────────
# renewal_feat = br.groupby(
#     ['Co_Ref','Prospect_Renewal_Date']
# ).agg(
#     renewal_call_count_14_45=('Call_Date','count'),
#     renewal_last_call_gap=('Days_Before_Renewal','min')
# ).reset_index()

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: CREATE CC FEATURES
# # ────────────────────────────────────────────────────────────────────
# cc_feat = bcc.groupby(
#     ['Co_Ref','Prospect_Renewal_Date']
# ).agg(
#     cc_call_count_14_45=('Call_Date','count'),
#     cc_last_call_gap=('Days_Before_Renewal','min')
# ).reset_index()

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 5: MERGE ALL DATA
# # ────────────────────────────────────────────────────────────────────
# final_df = billings_df.copy()

# # Merge renewal features
# final_df = final_df.merge(
#     renewal_feat,
#     on=['Co_Ref','Prospect_Renewal_Date'],
#     how='left'
# )

# # Merge cc features
# final_df = final_df.merge(
#     cc_feat,
#     on=['Co_Ref','Prospect_Renewal_Date'],
#     how='left'
# )

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 6: HANDLE MISSING VALUES
# # ────────────────────────────────────────────────────────────────────

# # Counts → 0 (no calls)
# final_df['renewal_call_count_14_45'] = final_df['renewal_call_count_14_45'].fillna(0)
# final_df['cc_call_count_14_45'] = final_df['cc_call_count_14_45'].fillna(0)

# # Gaps → large number (no interaction)
# final_df['renewal_last_call_gap'] = final_df['renewal_last_call_gap'].fillna(999)
# final_df['cc_last_call_gap'] = final_df['cc_last_call_gap'].fillna(999)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 7: FINAL CHECKS
# # ────────────────────────────────────────────────────────────────────
# print("Final dataset shape:", final_df.shape)
# print("\nSample data:")
# print(final_df.head())

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 8: SAVE FINAL DATASET
# # ────────────────────────────────────────────────────────────────────
# final_df.to_csv("../../dataset/05_merged/final_features.csv", index=False)

# print("\nSaved: final_features.csv")

### inner cc
Final dataset shape: (27197, 73)

### left cc 
Final dataset shape: (27197, 73)

In [161]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
# cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# # Keep required columns
# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], dayfirst=True, errors='coerce')
# cc_calls['Call_Date']      = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Remove invalid dates
# renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
# cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: BILLING + RENEWAL (INNER JOIN)
# # ────────────────────────────────────────────────────────────────────
# x = billings_df.merge(renewal_calls, on='Co_Ref', how='left')
# # x = billings_df.merge(renewal_calls, on='Co_Ref', how='inner')

# print("Shape after billing + renewal (INNER):", x.shape)

# # Keep only calls BEFORE renewal
# x = x[x['Call_Date'] <= x['Prospect_Renewal_Date']]

# # Days before renewal
# x['Days_Before_Renewal'] = (
#     x['Prospect_Renewal_Date'] - x['Call_Date']
# ).dt.days

# # Apply 14–45 filter (renewal calls)
# x = x[
#     (x['Days_Before_Renewal'] >= 14) &
#     (x['Days_Before_Renewal'] <= 45)
# ]

# print("Shape after renewal 14–45 filter:", x.shape)

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: ADD CC CALLS (LEFT JOIN)
# # ────────────────────────────────────────────────────────────────────
# merged_cc = x.merge(cc_calls, on='Co_Ref', how='left', suffixes=('_renewal', '_cc'))

# print("Shape after adding cc_calls (LEFT):", merged_cc.shape)

# # ── FILTER CC CALLS ─────────────────────────────────────────────────

# # Keep cc calls before renewal OR no call
# merged_cc = merged_cc[
#     (merged_cc['Call_Date_cc'].isna()) |
#     (merged_cc['Call_Date_cc'] <= merged_cc['Prospect_Renewal_Date'])
# ]

# # Calculate cc days before renewal
# merged_cc['cc_Days_Before_Renewal'] = (
#     merged_cc['Prospect_Renewal_Date'] - merged_cc['Call_Date_cc']
# ).dt.days

# # Apply 14–45 filter to cc calls ALSO
# merged_cc = merged_cc[
#     (merged_cc['cc_Days_Before_Renewal'].isna()) |
#     (
#         (merged_cc['cc_Days_Before_Renewal'] >= 14) &
#         (merged_cc['cc_Days_Before_Renewal'] <= 45)
#     )
# ]

# # Optional flag (still useful)
# merged_cc['cc_14_45_flag'] = (
#     (merged_cc['cc_Days_Before_Renewal'] >= 14) &
#     (merged_cc['cc_Days_Before_Renewal'] <= 45)
# ).astype(int)

# # ────────────────────────────────────────────────────────────────────
# # 📊 FINAL OUTPUT
# # ────────────────────────────────────────────────────────────────────
# print("\nFinal dataset shape:", merged_cc.shape)

# print("\nSample data after full merge:")
# print(merged_cc.head())

# # Optional: check distribution
# print("\nUnique customers:", merged_cc['Co_Ref'].nunique())
# print("\nOutcome distribution:")
# print(merged_cc['Prospect_Outcome'].value_counts())









# # //NEED TO DO AGG

# merged_cc.to_csv("../../dataset/05_merged/brc_merged.csv")


# Merged  (billing + renew (inner ) + cc (left)) + email


In [162]:
# import pandas as pd

# # ── LOAD DATA ───────────────────────────────────────────────────────
# renewal_calls = pd.read_csv('../../dataset/02_basic_clean/renewal_calls.csv')
# cc_calls      = pd.read_csv('../../dataset/02_basic_clean/cc_calls.csv')
# emails        = pd.read_csv('../../dataset/02_basic_clean/emails.csv')
# billings_df   = pd.read_csv('../../dataset/02_basic_clean/billings.csv', low_memory=False)

# # Keep required columns
# billings_df = billings_df[['Co_Ref', 'Prospect_Renewal_Date', 'Prospect_Outcome']]

# # ── DATE CONVERSION ─────────────────────────────────────────────────
# renewal_calls['Call_Date'] = pd.to_datetime(renewal_calls['Call_Date'], dayfirst=True, errors='coerce')
# cc_calls['Call_Date']      = pd.to_datetime(cc_calls['Call_Date'], dayfirst=True, errors='coerce')
# billings_df['Prospect_Renewal_Date'] = pd.to_datetime(
#     billings_df['Prospect_Renewal_Date'], dayfirst=True, errors='coerce'
# )

# # Clean
# renewal_calls = renewal_calls.dropna(subset=['Call_Date'])
# cc_calls      = cc_calls.dropna(subset=['Call_Date'])

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 1: BILLING + RENEWAL (LEFT JOIN)
# # ────────────────────────────────────────────────────────────────────
# x = billings_df.merge(renewal_calls, on='Co_Ref', how='left')

# # Keep only valid calls
# x = x[x['Call_Date'] <= x['Prospect_Renewal_Date']]

# # Days before renewal
# x['Days_Before_Renewal'] = (
#     x['Prospect_Renewal_Date'] - x['Call_Date']
# ).dt.days

# # Apply 14–45 filter
# x = x[
#     (x['Days_Before_Renewal'] >= 14) &
#     (x['Days_Before_Renewal'] <= 45)
# ]

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 2: ADD CC CALLS
# # ────────────────────────────────────────────────────────────────────
# merged_cc = x.merge(cc_calls, on='Co_Ref', how='left', suffixes=('_renewal', '_cc'))

# # Keep valid cc calls
# merged_cc = merged_cc[
#     (merged_cc['Call_Date_cc'].isna()) |
#     (merged_cc['Call_Date_cc'] <= merged_cc['Prospect_Renewal_Date'])
# ]

# # cc timing
# merged_cc['cc_Days_Before_Renewal'] = (
#     merged_cc['Prospect_Renewal_Date'] - merged_cc['Call_Date_cc']
# ).dt.days

# # 14–45 filter
# merged_cc = merged_cc[
#     (merged_cc['cc_Days_Before_Renewal'].isna()) |
#     (
#         (merged_cc['cc_Days_Before_Renewal'] >= 14) &
#         (merged_cc['cc_Days_Before_Renewal'] <= 45)
#     )
# ]

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 3: EMAIL PROCESSING
# # ────────────────────────────────────────────────────────────────────

# # ✔ Filter only 14 day emails
# email_filter = emails[emails['Time_to_Renewal'] == '14_out']

# # ✔ Aggregate emails per customer
# email_agg = email_filter.groupby('Co_Ref').agg(
#     email_count=('Co_Ref', 'count')
# ).reset_index()

# # ────────────────────────────────────────────────────────────────────
# # ✅ STEP 4: MERGE EMAILS
# # ────────────────────────────────────────────────────────────────────
# final_df = merged_cc.merge(email_agg, on='Co_Ref', how='left')

# # Fill missing emails
# final_df['email_count'] = final_df['email_count'].fillna(0)

# # ────────────────────────────────────────────────────────────────────
# # 📊 OUTPUT
# # ────────────────────────────────────────────────────────────────────
# print("\nFinal dataset shape:", final_df.shape)

# print("\nSample data:")
# print(final_df.head())

# print("\nUnique customers:", final_df['Co_Ref'].nunique())

# print("\nOutcome distribution:")
# print(final_df['Prospect_Outcome'].value_counts())




# # //NEED TO AGGREGATE


# final_df.to_csv("../../dataset/05_merged/brce_merged.csv")
